# 🕵️ Dia 84 — Reconhecimento da infraestrutura externa do caso

**Investigador Cyber 90 Dias** · Fase 7 · Rank Jutsu `S`

> Baseado no *Day 84* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-7-capstone-career/day-84.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).
> **Material sintético.** O *case packet* (Orrin Valley / Castellan) e a Operação Isca são ficção; domínios `.example`, IPs de documentação.

---

### 🎬 Onde paramos

Rafa deixou na sua mesa nove pivôs da Operação Isca e uma pergunta: *"Quantas dessas linhas aguentam um advogado do outro lado?"* No capstone, o ponto de partida é o e-mail de phishing de 10 de março.

Neste notebook você vai:

1. **Ler os cabeçalhos `Received` de baixo para cima** com o módulo `email`.
2. **Desmascarar o link**: texto visível × `href`, e os parâmetros em Base64.
3. **Gerar a tabela de indicadores desarmada.**
4. **Comparar o site real com o clone** com `difflib`.
5. **Testar a explicação inocente** de cada IP (o mesmo /24 não liga ninguém).
6. **Jogar o quiz dos pivôs**: achado, pista ou rejeitado (capstone + Isca).

🐍 Só biblioteca padrão do Python. Funciona offline: se a rede falhar, o notebook usa a cópia embutida dos arquivos. Rode as células de cima para baixo (Shift+Enter).

## 📦 O pacote do caso (plano A: GitHub; plano B: cópia embutida)

A próxima célula guarda uma **cópia embutida** dos arquivos do *case packet* que este dia usa. Ela é o plano B: se você estiver sem internet, ou se o GitHub estiver fora, o notebook continua funcionando.

A célula seguinte tenta o plano A: baixar cada arquivo do repositório original com `urllib`. Se baixar, compara com a cópia embutida pelo SHA-256 e avisa se o original mudou desde que este notebook foi gerado (é exatamente o raciocínio de custódia do Dia 83).

In [ ]:
# Cópia embutida (plano B). Cada valor é o texto exato do arquivo do case packet.
# Não edite: se mudar um caractere, o SHA-256 muda e a custódia quebra.
EMBUTIDO = {
"02-suspicious-email.eml": "X-Case-Packet-Notice: SYNTHETIC TRAINING DATA. Fictional organizations, people, domains (RFC 2606 .example) and IPs (RFC 5737). Do not look up or contact anything in this file.\nReturn-Path: <bounce-7731@castellan-hardwood.example>\nReceived: from gw01.orrinvalley.example (gw01.orrinvalley.example [192.0.2.25])\n\tby mbx01.orrinvalley.example with ESMTPS id 4f1c9a2e\n\tfor <jpike@orrinvalley.example>; Tue, 10 Mar 2026 14:02:39 +0000\nReceived: from mail.castellan-hardwood.example (vps-47.pinecrate.example [203.0.113.47])\n\tby gw01.orrinvalley.example with ESMTPS id 8b3e0d17\n\tfor <jpike@orrinvalley.example>; Tue, 10 Mar 2026 14:02:37 +0000\nReceived: from [10.8.0.6] (unknown [198.51.100.23])\n\t(Authenticated sender: ar@castellan-hardwood.example)\n\tby mail.castellan-hardwood.example (Postfix) with ESMTPSA id 2C4A1E03\n\tfor <jpike@orrinvalley.example>; Tue, 10 Mar 2026 14:02:35 +0000\nAuthentication-Results: gw01.orrinvalley.example;\n\tspf=pass (gw01.orrinvalley.example: domain of bounce-7731@castellan-hardwood.example designates 203.0.113.47 as permitted sender) smtp.mailfrom=castellan-hardwood.example;\n\tdkim=none (message not signed);\n\tdmarc=none action=none header.from=castellan-hardwood.example\nX-OVM-Gateway-Verdict: clean; score=3.1; threshold=5.0; tags=EXTERNAL\nDate: Tue, 10 Mar 2026 10:02:31 -0400\nFrom: \"Dana Whitlock | Castellan Hardwood Supply\" <dana.whitlock@castellan-hardwood.example>\nReply-To: <dwhitlock.castellan@freemail.example>\nTo: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] RE: Invoice CHS-20417 - updated remittance details\nMessage-ID: <20260310140231.2C4A1E03@mail.castellan-hardwood.example>\nIn-Reply-To: <20260303144702.GA11873@mail.castellanhardwood.example>\nReferences: <20260303144702.GA11873@mail.castellanhardwood.example>\nUser-Agent: Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Thunderbird/115.8.0\nMIME-Version: 1.0\nContent-Type: multipart/alternative; boundary=\"------------b7Qx2LmR0v8aZ3\"\n\nThis is a multi-part message in MIME format.\n--------------b7Qx2LmR0v8aZ3\nContent-Type: text/plain; charset=UTF-8\nContent-Transfer-Encoding: 7bit\n\nHi Jordan,\n\nFollowing up on the invoice below. As part of our annual audit we have\nmoved our receivables to a new account, effective this week. Please use\nthe updated remittance details for CHS-20417 and for all future payments.\nThe signed bank letter and the vendor update form are on our secure\nportal:\n\nView remittance form (secure): https://castellanhardwood.example/remittance\n\nOur office phones are being migrated to a new system this week, so email\nis the quickest way to reach me. We would appreciate it if CHS-20417 could\ngo out in Friday's payment run, as the old account closes on the 16th.\n\nThanks so much,\n\nDana Whitlock\nAccounts Receivable Manager\nCastellan Hardwood Supply\nTel. 555-0148 | ar@castellan-hardwood.example\n\nOn Tue, Mar 3, 2026 at 9:47 AM Dana Whitlock <dwhitlock@castellanhardwood.example> wrote:\n> Good morning,\n>\n> Please find attached invoice CHS-20417 for the February white oak and\n> hard maple order (PO 7719), total $48,612.50, net 10.\n>\n> Thank you for your business.\n>\n> Dana Whitlock\n> Accounts Receivable Manager\n> Castellan Hardwood Supply\n> Tel. 555-0112 | accounts@castellanhardwood.example\n\n--------------b7Qx2LmR0v8aZ3\nContent-Type: text/html; charset=UTF-8\nContent-Transfer-Encoding: 7bit\n\n<html><body style=\"font-family:Calibri,Arial,sans-serif;font-size:11pt\">\n<p>Hi Jordan,</p>\n<p>Following up on the invoice below. As part of our annual audit we have moved our receivables to a new account, effective this week. Please use the updated remittance details for CHS-20417 and for all future payments. The signed bank letter and the vendor update form are on our secure portal:</p>\n<p><b>View remittance form (secure):</b> <a href=\"https://cstl-docshare.example/view/r?u=b3JyaW52YWxsZXkuZXhhbXBsZQ%3D%3D&amp;t=anBpa2U%3D\">https://castellanhardwood.example/remittance</a></p>\n<p>Our office phones are being migrated to a new system this week, so email is the quickest way to reach me. We would appreciate it if CHS-20417 could go out in Friday's payment run, as the old account closes on the 16th.</p>\n<p>Thanks so much,</p>\n<p>Dana Whitlock<br>Accounts Receivable Manager<br>Castellan Hardwood Supply<br>Tel. 555-0148 | ar@castellan-hardwood.example</p>\n<img src=\"https://castellanhardwood.example/img/logo.png\" width=\"160\" alt=\"Castellan Hardwood Supply\">\n<div style=\"border-left:2px solid #ccc;padding-left:8px;color:#555\">\n<p>On Tue, Mar 3, 2026 at 9:47 AM Dana Whitlock &lt;dwhitlock@castellanhardwood.example&gt; wrote:</p>\n<p>Good morning,</p>\n<p>Please find attached invoice CHS-20417 for the February white oak and hard maple order (PO 7719), total $48,612.50, net 10.</p>\n<p>Thank you for your business.</p>\n<p>Dana Whitlock<br>Accounts Receivable Manager<br>Castellan Hardwood Supply<br>Tel. 555-0112 | accounts@castellanhardwood.example</p>\n</div>\n</body></html>\n\n--------------b7Qx2LmR0v8aZ3--\n",
"03-related-messages.md": "# Related messages\n\n> SYNTHETIC TRAINING MATERIAL. All organizations, people, domains, IPs and account numbers are fictional.\n\nThe client's IT contractor pulled these messages from mailboxes on 16 and 17 March 2026. Headers are trimmed to the fields that matter; the full suspicious email is in `02-suspicious-email.eml`. Message IDs (M1 to M7) are yours to use in notes and on the case graph.\n\n## M1: the real invoice (3 March)\n\nWhere found: shared mailbox `ap@orrinvalley.example`, Inbox.\n\n```\nDate: Tue, 3 Mar 2026 09:47:02 -0500\nFrom: Dana Whitlock <dwhitlock@castellanhardwood.example>\nTo: ap@orrinvalley.example\nCc: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] Invoice CHS-20417\nMessage-ID: <20260303144702.GA11873@mail.castellanhardwood.example>\nAuthentication-Results: gw01.orrinvalley.example; spf=pass smtp.mailfrom=castellanhardwood.example;\n\tdkim=pass header.d=castellanhardwood.example; dmarc=pass (p=quarantine) header.from=castellanhardwood.example\nAttachment: CHS-20417.pdf\n```\n\n```\nGood morning,\n\nPlease find attached invoice CHS-20417 for the February white oak and\nhard maple order (PO 7719), total $48,612.50, net 10.\n\nThank you for your business.\n\nDana Whitlock\nAccounts Receivable Manager\nCastellan Hardwood Supply\nTel. 555-0112 | accounts@castellanhardwood.example\n```\n\nThe attached invoice PDF names the payee as \"Castellan Hardwood Supply Inc.\" with remittance to Vellacourt Commercial Bank, account ending 4471. It has been Castellan's account on file since 2019.\n\n## M2: the suspicious email (10 March)\n\nSee `02-suspicious-email.eml`.\n\n## M3: \"Jordan\" to the controller (11 March)\n\nWhere found: `mbeaulieu` Inbox. No copy exists in Jordan's Sent Items or Deleted Items.\n\n```\nDate: Wed, 11 Mar 2026 09:52:44 -0400\nFrom: Jordan Pike <jpike@orrinvalley.example>\nTo: Maren Beaulieu <mbeaulieu@orrinvalley.example>\nSubject: Castellan banking update - vendor change for Friday run\nMessage-ID: <a81f3c07e2d94b6e@mbx01.orrinvalley.example>\nAttachment: Castellan_Bank_Letter_2026.pdf\n```\n\n```\nHi Maren,\n\nCastellan sent over their updated banking letter (attached) - they moved\nbanks as part of their annual audit and the old account closes on the 16th.\nCan you update the vendor master before Friday's run? They also asked that\nremittance advices go to ar@castellan-hardwood.example from now on.\n\nDana mentioned their phones are down this week while they switch systems,\nso no need to call, she said email is best.\n\nThanks!\nJordan\n```\n\nText of the attachment, `Castellan_Bank_Letter_2026.pdf`, as extracted by IT:\n\n```\nCASTELLAN HARDWOOD SUPPLY LTD.\n1400 Ferrule Road, Wrenmoor Crossing  |  Tel. 555-0148  |  ar@castellan-hardwood.example\n\n9 March 2026\n\nTo our valued customers,\n\nRE: CHANGE OF BANKING DETAILS\n\nFollowing our annual financial audit, Castellan Hardwood Supply has\ntransferred its receivables to a new banking partner. Effective immediately,\nplease remit all payments to:\n\n    Bank:            Pellbrook Savings & Trust\n    Account name:    Castellan Hardwood Supply Ltd.\n    Account ending:  8830\n\nOur previous account will be closed on 16 March 2026. Payments sent to the\nprevious account after that date will be returned and may incur late fees.\n\nPlease direct any questions to our Accounts Receivable team at\nar@castellan-hardwood.example.\n\nSincerely,\n\nDana Whitlock                          G. Castellan\nAccounts Receivable Manager            Chief Financial Officer\n```\n\nDocument metadata for the attachment, from `exiftool Castellan_Bank_Letter_2026.pdf` run by IT:\n\n```\nFile Size                       : 41 kB\nPDF Version                     : 1.7\nPage Count                      : 1\nProducer                        : LibreOffice 7.6\nCreator                         : Writer\nAuthor                          : user\nTitle                           : vendor_letter_v3\nCreate Date                     : 2026:02:26 19:12:44-05:00\nModify Date                     : 2026:03:09 22:41:03-04:00\n```\n\nSHA-256 (as recorded by the mail gateway): `5d0c7a1e9b43f26c8e7a0d51b9f4c3e2a6d8b17f04e9c5a2d3b6f8e1c7a9042b`\n\n## M4: the controller's reply (11 March)\n\nWhere found: `mbeaulieu` Sent Items, and Jordan's `RSS Subscriptions` folder, marked as read.\n\n```\nDate: Wed, 11 Mar 2026 11:06:12 -0400\nFrom: Maren Beaulieu <mbeaulieu@orrinvalley.example>\nTo: Jordan Pike <jpike@orrinvalley.example>\nSubject: RE: Castellan banking update - vendor change for Friday run\n```\n\n```\nDone, updated in the vendor master (bank + remittance email). Please save\nthe letter to the Castellan vendor file. M.\n```\n\n## M5: the real reminder (12 March)\n\nWhere found: Jordan's `RSS Subscriptions` folder, marked as read. The copy in `ap@` Inbox is unread.\n\n```\nDate: Thu, 12 Mar 2026 12:05:38 -0400\nFrom: Dana Whitlock <dwhitlock@castellanhardwood.example>\nTo: ap@orrinvalley.example\nCc: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] Reminder: CHS-20417 due 13 March\nAuthentication-Results: gw01.orrinvalley.example; spf=pass smtp.mailfrom=castellanhardwood.example;\n\tdkim=pass header.d=castellanhardwood.example; dmarc=pass (p=quarantine) header.from=castellanhardwood.example\n```\n\n```\nHi Jordan,\n\nFriendly reminder that CHS-20417 ($48,612.50) is due tomorrow. Our\nbanking details are unchanged from the invoice.\n\nThanks,\nDana\nTel. 555-0112\n```\n\n## M6: the attacker's follow-up (12 March)\n\nWhere found: Jordan's `RSS Subscriptions` folder, marked as read.\n\n```\nDate: Thu, 12 Mar 2026 12:39:58 -0400\nFrom: \"Dana Whitlock | Castellan Hardwood Supply\" <dana.whitlock@castellan-hardwood.example>\nReply-To: <dwhitlock.castellan@freemail.example>\nTo: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] RE: Invoice CHS-20417 - updated remittance details\nAuthentication-Results: gw01.orrinvalley.example; spf=pass smtp.mailfrom=castellan-hardwood.example;\n\tdkim=none; dmarc=none header.from=castellan-hardwood.example\n```\n\n```\nHi Jordan,\n\nJust confirming the update went through on your side. Could you send the\nremittance advice as soon as the payment is released tomorrow? Our auditors\nneed it for the account transition.\n\nThanks again for your help with this,\nDana\n```\n\n## M7: the automated remittance advice (13 March)\n\nWhere found: mail gateway trace only (outbound). The ERP sends remittance advices automatically to the remittance address on the vendor record.\n\n```\nDate: Fri, 13 Mar 2026 15:02:07 -0400\nFrom: Orrin Valley Millworks Payables <erp-noreply@orrinvalley.example>\nTo: ar@castellan-hardwood.example\nSubject: Remittance advice - Orrin Valley Millworks - payment 000731\n```\n\n```\nPayment 000731 has been released.\nPayee: Castellan Hardwood Supply Ltd.   Amount: $48,612.50\nInvoices: CHS-20417\nRemitted to: Pellbrook Savings & Trust, account ending 8830\n```\n",
"04-vendor-site-archived.md": "# Artifact A: Castellan Hardwood Supply \"About us\" page (archived)\n\n> SYNTHETIC TRAINING MATERIAL. Castellan Hardwood Supply, Wrenmoor Crossing and every person named here are fictional.\n\nSource: web archive capture of `https://castellanhardwood.example/about`, captured 2 November 2025 at 16:20 UTC. The client's IT contractor saved the rendered text and the page head on 17 March 2026. This is the genuine vendor's site as it looked before the incident.\n\n## Page head (excerpt)\n\n```html\n<title>About Us | Castellan Hardwood Supply</title>\n<meta name=\"description\" content=\"Family-owned hardwood lumber yard supplying cabinetmakers and furniture builders since 1986.\">\n<link rel=\"icon\" href=\"/favicon.ico\">\n<link rel=\"stylesheet\" href=\"/css/site.css?v=2025.3\">\n```\n\n## Rendered text\n\n> **About Castellan Hardwood Supply**\n>\n> Castellan Hardwood Supply has milled and supplied North American hardwoods to cabinetmakers, millwork shops and furniture builders since 1986. We are a family-owned business, now in our third generation, operating from our yard and kiln facility at 1400 Ferrule Road in Wrenmoor Crossing.\n>\n> We stock white oak, red oak, hard and soft maple, cherry, walnut and ash in 4/4 through 12/4, with custom milling to your cut list. Most orders within our delivery region ship within five business days.\n>\n> **Our team**\n>\n> Graham Castellan, President\n> Ines Castellan-Rowe, Chief Financial Officer\n> Tobias Venn, Yard and Kiln Manager\n> Dana Whitlock, Accounts Receivable Manager\n>\n> **Contact**\n>\n> Orders: orders@castellanhardwood.example\n> Accounts: accounts@castellanhardwood.example\n> Phone: 555-0112 (Monday to Friday, 7:00 to 16:00)\n>\n> **Payments**\n>\n> For your security, Castellan Hardwood Supply will never change its banking details by email alone. If you receive a request to change how you pay us, call our office at the number above before acting on it.\n>\n> © 2025 Castellan Hardwood Supply Inc.\n",
"05-lookalike-site-capture.md": "# Artifact B: the lookalike site (captured)\n\n> SYNTHETIC TRAINING MATERIAL. All organizations, people, domains and IPs are fictional.\n\nSource: the client's IT contractor captured these pages on 17 March 2026 at 13:05 UTC from an isolated, sandboxed browser that the contractor's firm uses for this purpose. The capture is the authorized collection for this case. You do not need to visit the site, and the engagement letter forbids it.\n\n## B1: `https://castellan-hardwood.example/about`\n\nPage head (excerpt):\n\n```html\n<title>About Us | Castellan Hardwood Supply</title>\n<meta name=\"description\" content=\"Family-owned hardwood lumber yard supplying cabinetmakers and furniture builders since 1986.\">\n<link rel=\"icon\" href=\"/favicon.ico\">\n<link rel=\"stylesheet\" href=\"https://castellanhardwood.example/css/site.css?v=2024.1\">\n<img class=\"logo\" src=\"https://castellanhardwood.example/img/logo.png\" alt=\"Castellan Hardwood Supply\">\n```\n\nRendered text:\n\n> **About Castellan Hardwood Supply**\n>\n> Castellan Hardwood Supply has milled and supplied North American hardwoods to cabinetmakers, millwork shops and furniture builders since 1986. We are a family-owned business, now in our third generation, operating from our yard and kiln facility at 1400 Ferrule Road in Wrenmoor Crossing.\n>\n> We stock white oak, red oak, hard and soft maple, cherry, walnut and ash in 4/4 through 12/4, with custom milling to your cut list. Most orders within our delivery region ship within five business days.\n>\n> **Banking update**\n>\n> Following our annual audit, Castellan Hardwood Supply has moved to a new banking partner as of March 2026. Existing customers will receive updated remittance instructions from our Accounts Receivable team. Please contact ar@castellan-hardwood.example with any questions.\n>\n> **Our team**\n>\n> Graham Castellan, President\n> Tobias Venn, Yard and Kiln Manager\n> Dana Whitlock, Accounts Receivable Manager\n>\n> **Contact**\n>\n> Orders: orders@castellan-hardwood.example\n> Accounts: ar@castellan-hardwood.example\n> Phone: 555-0148 (Monday to Friday, 7:00 to 16:00)\n>\n> © 2024 Castellan Hardwood Supply Inc.\n\n## B2: `https://castellan-hardwood.example/privacy`\n\nRendered text (complete):\n\n> **Privacy policy**\n>\n> Castellan Hardwood Supply collects only the information needed to process your orders and payments. We do not sell your information to third parties. Marrowline Freight Services respects your privacy and will only use your contact details to respond to your enquiry. For questions about this policy, contact ar@castellan-hardwood.example.\n\n## B3: other paths requested during the capture\n\n| Path | Status | Notes |\n|---|---|---|\n| `/` | 200 | Home page. Same layout as the real site; the hero image is hotlinked from `castellanhardwood.example`. |\n| `/contact` | 200 | Contact form. The form posts to `/assets/php/contact.php`. |\n| `/remittance` | 404 | No page at the path shown in the phishing email's link text. |\n| `/favicon.ico` | 200 | Byte-identical to the real site's favicon (see the hash in `06-lookup-results.md`). |\n",
"06-lookup-results.md": "# Pre-collected lookup results\n\n> SYNTHETIC TRAINING MATERIAL. Every domain uses the reserved `.example` name, every IP is an RFC 5737 documentation address, and every AS number is an RFC 5398 documentation ASN. Registrars, hosting companies and carriers named here are fictional.\n\nThe client's IT contractor ran these passive lookups on 17 March 2026 between 12:40 and 13:30 UTC, using WHOIS, a passive DNS service, a certificate transparency search, a URL scanning service and a favicon-hash search. In a real case you would run them yourself and log each one. Here, reading a section counts as running that lookup: log it in your collection log with the section number as the source and 17 March 2026 as the access date.\n\nSection numbers (L1 to L9) are for citation in your notes and case graph.\n\n## L1: WHOIS\n\n| Field | castellanhardwood.example | castellan-hardwood.example | cstl-docshare.example | marrowline-docs.example | quennmarsh-accounts.example |\n|---|---|---|---|---|---|\n| Created | 2004-06-14 | 2026-02-24T21:14:09Z | 2026-02-26T03:40:51Z | 2026-02-19T22:05:33Z | 2026-03-14T02:51:17Z |\n| Registrar | Beaconhill Domains | Quillfeather Names | Quillfeather Names | Quillfeather Names | Quillfeather Names |\n| Registrant org | Castellan Hardwood Supply Inc. | REDACTED FOR PRIVACY | REDACTED FOR PRIVACY | REDACTED FOR PRIVACY | REDACTED FOR PRIVACY |\n| Name servers | ns1/ns2.cedarlinedns.example | ns1/ns2.hollowdns.example | ns1/ns2.hollowdns.example | ns1/ns2.hollowdns.example | ns1/ns2.hollowdns.example |\n| Expires | 2027-06-14 | 2027-02-24 | 2027-02-26 | 2027-02-19 | 2027-03-14 |\n\nContext from the registrar and DNS host's own public pages: Quillfeather Names is a low-cost registrar with about 3 million domains under management. Hollowdns is a free DNS hosting service with about 90,000 zones.\n\n## L2: current DNS records (17 March 2026)\n\n```\ncastellanhardwood.example.          A      192.0.2.140\ncastellanhardwood.example.          MX     10 mail.castellanhardwood.example.\nmail.castellanhardwood.example.     A      198.51.100.180\ncastellanhardwood.example.          TXT    \"v=spf1 ip4:198.51.100.180 -all\"\n_dmarc.castellanhardwood.example.   TXT    \"v=DMARC1; p=quarantine; rua=mailto:dmarc@castellanhardwood.example\"\n\ncastellan-hardwood.example.         A      203.0.113.47\ncastellan-hardwood.example.         MX     10 mail.castellan-hardwood.example.\nmail.castellan-hardwood.example.    A      203.0.113.47\ncastellan-hardwood.example.         TXT    \"v=spf1 ip4:203.0.113.47 -all\"\n_dmarc.castellan-hardwood.example.  NXDOMAIN\n\ncstl-docshare.example.              A      203.0.113.88\nwww.cstl-docshare.example.          CNAME  cstl-docshare.example.\n\nmarrowline-docs.example.            A      203.0.113.91\nportal.marrowline-docs.example.     A      203.0.113.91\nmail.marrowline-docs.example.       NXDOMAIN\n\nquennmarsh-accounts.example.        A      203.0.113.91\n```\n\n## L3: passive DNS history\n\n| rrname | Type | rdata | First seen (UTC) | Last seen (UTC) |\n|---|---|---|---|---|\n| castellanhardwood.example | A | 192.0.2.140 | 2019-04-02 | 2026-03-17 |\n| mail.castellanhardwood.example | A | 198.51.100.170 | 2017-08-11 | 2026-01-19 |\n| mail.castellanhardwood.example | A | 198.51.100.180 | 2026-01-19 | 2026-03-17 |\n| castellan-hardwood.example | A | 203.0.113.47 | 2026-02-25 | 2026-03-17 |\n| mail.castellan-hardwood.example | A | 203.0.113.47 | 2026-02-25 | 2026-03-17 |\n| cstl-docshare.example | A | 203.0.113.88 | 2026-02-27 | 2026-03-17 |\n| marrowline-docs.example | A | 203.0.113.91 | 2026-02-20 | 2026-03-17 |\n| portal.marrowline-docs.example | A | 203.0.113.91 | 2026-02-20 | 2026-03-17 |\n| mail.marrowline-docs.example | A | 198.51.100.23 | 2026-02-20 | 2026-03-09 |\n| quennmarsh-accounts.example | A | 203.0.113.91 | 2026-03-15 | 2026-03-17 |\n\nReverse lookups on the same service:\n\n| IP | Hostnames seen resolving to it | PTR record |\n|---|---|---|\n| 198.51.100.23 | mail.marrowline-docs.example (2026-02-20 to 2026-03-09) | vps-23.pinecrate.example |\n| 203.0.113.47 | castellan-hardwood.example, mail.castellan-hardwood.example | vps-47.pinecrate.example |\n| 203.0.113.91 | marrowline-docs.example, portal.marrowline-docs.example, quennmarsh-accounts.example | none |\n\n## L4: certificate transparency\n\n| Log entry ID | Names on certificate | Issuer | Not before (UTC) |\n|---|---|---|---|\n| 9048820415 | marrowline-docs.example, portal.marrowline-docs.example | Let's Encrypt | 2026-02-20 01:37 |\n| 9048820911 | mail.marrowline-docs.example | Let's Encrypt | 2026-02-20 01:39 |\n| 9051182231 | castellan-hardwood.example, www.castellan-hardwood.example, mail.castellan-hardwood.example | Let's Encrypt | 2026-02-25 06:02 |\n| 9051497710 | cstl-docshare.example, www.cstl-docshare.example | Let's Encrypt | 2026-02-27 04:11 |\n| 9063310054 | quennmarsh-accounts.example | Let's Encrypt | 2026-03-15 00:22 |\n| (routine) | castellanhardwood.example, www.castellanhardwood.example | Let's Encrypt | Renewed every 60 to 90 days since 2019; latest 2025-12-30 |\n\n## L5: co-hosting (reverse IP)\n\n| IP | Hosting | Domains currently hosted | Sample |\n|---|---|---|---|\n| 203.0.113.47 | Pinecrate VPS, single-tenant virtual server | 2 hostnames | See L3 |\n| 203.0.113.88 | Pinecrest Shared Hosting, shared web server | 1,412 domains | ferncastle-bakery.example, oakhollow-dental.example, rivermint-yoga.example, tidepost-news.example, cstl-docshare.example, (1,407 more) |\n| 203.0.113.91 | Driftwood Cloud VPS, single-tenant virtual server | 3 hostnames | See L3 |\n\n## L6: URL scan results\n\nS1. Scanned by the client's IT contractor, 17 March 2026 13:20 UTC, sandboxed scanner.\n\n```\nURL:         https://cstl-docshare.example/view/r?u=b3JyaW52YWxsZXkuZXhhbXBsZQ%3D%3D&t=anBpa2U%3D\nFinal URL:   (same)            Status: 200        IP: 203.0.113.88\nPage title:  Secure Document Viewer\nPage text:   \"Castellan Hardwood Supply shared a secure document with you.\n              Sign in with your work email to view Remittance_Form.pdf\"\nResources:   /assets/js/vw-portal.min.js   sha256 9e41c0b7d2a85f3e16c47d09ab3f58e2c1d06b7a94e2f35c8b01d7e6a4c9f218\n             /assets/css/vw.css\n             /favicon.ico                  mmh3 -1742093313\n             https://castellanhardwood.example/img/logo.png   (third-party image load)\nForm:        POST /view/auth   fields: email (prefilled \"jpike@orrinvalley.example\"), password\nNotes:       https://cstl-docshare.example/ returns 404. Scanner followed no redirect.\n```\n\nS2. From a public scan database. Submitted by an unknown third party on 22 February 2026 09:14 UTC.\n\n```\nURL:         https://portal.marrowline-docs.example/view/r?u=[redacted by database]\nStatus:      200        IP: 203.0.113.91\nPage title:  Secure Document Viewer\nPage text:   \"Marrowline Freight Services shared a secure document with you.\n              Sign in with your work email to view Updated_Payment_Instructions.pdf\"\nResources:   /assets/js/vw-portal.min.js   sha256 9e41c0b7d2a85f3e16c47d09ab3f58e2c1d06b7a94e2f35c8b01d7e6a4c9f218\n             /assets/css/vw.css\n             /favicon.ico                  mmh3 -1742093313\n             https://marrowlinefreight.example/images/brand.png   (third-party image load)\nForm:        POST /view/auth   fields: email, password\n```\n\nS3. From a public scan database. Submitted by an unknown third party on 8 October 2025 16:02 UTC.\n\n```\nURL:         https://files.brightquay-legal.example/\nStatus:      200        IP: 192.0.2.201\nPage title:  Secure Document Viewer | Brightquay Legal\nResources:   /static/viewer.js             sha256 3b7f0e92c4d15a8e6f20b9c7d41e5a3f8c62d09e1b4a7f35e8d0c6b2a91f4e07\n             /favicon.ico                  mmh3 -1742093313\nForm:        POST /sso/login\nPage footer: \"Built on the OpenDocViewer template\"\nCert history: certificates for files.brightquay-legal.example continuously since 2021\n```\n\n## L7: favicon hash search\n\n| mmh3 hash | Hosts returned |\n|---|---|\n| -1742093313 | cstl-docshare.example, portal.marrowline-docs.example, files.brightquay-legal.example |\n| 884215106 | castellanhardwood.example, castellan-hardwood.example |\n\n## L8: IP enrichment\n\n| IP | ASN | Network owner | Network type | Location (as reported by the geolocation service) | Other context |\n|---|---|---|---|---|---|\n| 192.0.2.10 | AS64496 | Wrenmoor Fibre | Business fibre | Client's metro area | Orrin Valley office NAT address (confirmed by IT) |\n| 192.0.2.25 | AS64496 | Wrenmoor Fibre | Business fibre | Client's metro area | Orrin Valley mail gateway gw01 (confirmed by IT) |\n| 192.0.2.140 | AS64500 | Cedarline Web Hosting | Managed hosting | Client's region | Castellan's real website |\n| 192.0.2.201 | AS64500 | Cedarline Web Hosting | Managed hosting | Client's region | Brightquay Legal document portal |\n| 198.51.100.23 | AS64510 | Pinecrate VPS | Datacenter / VPS | EU-West datacenter | None |\n| 198.51.100.77 | AS64508 | Skerrow Mobile | Mobile carrier, carrier-grade NAT | Client's metro area | Shared by many mobile subscribers |\n| 198.51.100.170 | AS64502 | Northfold Mail Services | Email hosting | Client's region | Castellan's previous mail server |\n| 198.51.100.180 | AS64502 | Northfold Mail Services | Email hosting | Client's region | Castellan's current mail server |\n| 203.0.113.47 | AS64510 | Pinecrate VPS | Datacenter / VPS | EU-West datacenter | None |\n| 203.0.113.88 | AS64505 | Pinecrest Shared Hosting | Shared web hosting | North America | 1,412 co-hosted domains (L5) |\n| 203.0.113.91 | AS64511 | Driftwood Cloud | Datacenter / VPS | North America | None |\n| 203.0.113.200 | AS64507 | Keelson Hosting | Datacenter | Asia-Pacific datacenter | Listed on a public blocklist for password spraying against cloud mail tenants since January 2026 |\n\nGeolocation for datacenter and VPS addresses reports where the server sits, which says nothing about where its operator is.\n\n## L9: other lookups\n\n| Item | Result |\n|---|---|\n| Phone 555-0112 | Landline. Carrier Wrenmoor Telephone. Directory listing: Castellan Hardwood Supply Inc., listed since 1998. |\n| Phone 555-0148 | VoIP. Carrier Callwisp VoIP. No directory listing. |\n| Domain `freemail.example` | Large consumer webmail provider. Anyone can register an address. No lookup was run on the address `dwhitlock.castellan@freemail.example` itself (out of scope). |\n| Public phishing feed | `marrowline-docs.example` reported 24 February 2026. `cstl-docshare.example` reported 12 March 2026 14:30 UTC by a reporter outside the client. `castellan-hardwood.example` and `quennmarsh-accounts.example` not listed. |\n"
}

In [ ]:
import hashlib                     # SHA-256: a impressão digital de cada arquivo
import urllib.request              # baixar sem instalar nada (requests não é stdlib)

RAW = "https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-7-capstone-career/case-packet/"   # endereço "cru" dos arquivos no GitHub

def baixar(nome, timeout=10):
    """Plano A: baixa o arquivo do GitHub. Devolve o texto ou None se falhar."""
    try:
        with urllib.request.urlopen(RAW + nome, timeout=timeout) as resp:   # timeout: não travar o Colab
            return resp.read().decode("utf-8")                              # bytes -> texto
    except Exception as erro:                                               # sem rede, 404, proxy...
        print(f"   (rede indisponível para {nome}: {type(erro).__name__}) -> uso a cópia embutida")
        return None

def sha(texto):
    """SHA-256 de um texto em UTF-8 (o mesmo que sha256sum daria no arquivo)."""
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

PACOTE = {}                                   # nome do arquivo -> texto que vamos analisar
for nome, copia in EMBUTIDO.items():
    online = baixar(nome)
    if online is None:                         # plano B
        PACOTE[nome] = copia
        origem = "embutido"
    else:                                      # plano A deu certo: confere com a cópia
        PACOTE[nome] = online
        origem = "GitHub" if sha(online) == sha(copia) else "GitHub (DIFERENTE da cópia embutida!)"
    print(f"{nome:32} {sha(PACOTE[nome])[:16]}…  [{origem}]")

## 1. Os cabeçalhos, de baixo para cima

Cada servidor acrescenta seu `Received` **por cima**. Então o primeiro salto real é o último cabeçalho do arquivo. O `email.parser` da biblioteca padrão devolve os cabeçalhos na ordem do arquivo; nós invertemos.

In [ ]:
from email import policy
from email.parser import Parser          # Parser (texto); BytesParser seria para bytes de arquivo

msg = Parser(policy=policy.default).parsestr(PACOTE["02-suspicious-email.eml"])

recebidos = msg.get_all("Received")      # lista, na ordem em que aparecem no arquivo
for salto, rec in enumerate(reversed(recebidos), start=1):   # reversed = ordem real de entrega
    compacto = " ".join(str(rec).split())                    # junta as linhas dobradas do cabeçalho
    print(f"Salto {salto}: {compacto}\n")

In [ ]:
import re

# Extraímos o essencial de cada salto com expressões regulares simples.
for salto, rec in enumerate(reversed(recebidos), start=1):
    r = " ".join(str(rec).split())
    ips = re.findall(r"\[(\d{1,3}(?:\.\d{1,3}){3})\]", r)          # IPs entre colchetes
    auth = re.search(r"Authenticated sender: ([^)]+)\)", r)           # submissão autenticada?
    quando = r.split(";")[-1].strip()                                  # a data vem depois do ";"
    print(f"Salto {salto}: IPs={ips}  autenticado={auth.group(1) if auth else '-'}  quando={quando}")

print("\nFrom    :", msg["From"])
print("Reply-To:", msg["Reply-To"], "  <- diferente do From: respostas vão para outro lugar")
print("In-Reply-To:", msg["In-Reply-To"], "  <- Message-ID do M1 verdadeiro")
print("\nAuthentication-Results:\n ", " ".join(str(msg["Authentication-Results"]).split()))

> 💡 **SPF passou. E daí?** O SPF só diz que `203.0.113.47` está autorizado a enviar pelo domínio **`castellan-hardwood.example`**, que é do atacante. Ele não diz nada sobre ser a Castellan. O que informa é o contraste: o M1 verdadeiro tem SPF, DKIM e DMARC passando para `castellanhardwood.example`; o phishing tem DKIM `none` e DMARC `none`.

## 2. O link: o que o Jordan viu × para onde o navegador ia

In [ ]:
import base64
import html
from urllib.parse import urlparse, parse_qs, unquote

corpo_html = msg.get_body(preferencelist=("html",)).get_content()      # a parte text/html
# Captura href e texto visível de cada <a ...>...</a>
for href, texto in re.findall(r'<a href="([^"]+)">([^<]+)</a>', corpo_html):
    href = html.unescape(href)                       # &amp; -> &
    print("Texto visível:", texto)
    print("href real    :", href)
    print("Domínios diferentes?", urlparse(texto).hostname != urlparse(href).hostname)
    params = parse_qs(urlparse(href).query)          # parse_qs já faz o URL decode (%3D -> =)
    for nome, (valor,) in params.items():
        decodificado = base64.b64decode(valor).decode()    # From Base64, como no CyberChef
        print(f"   {nome} = {valor:30} -> {decodificado}")

**Achado de direcionamento:** `u` decodifica para `orrinvalley.example` e `t` para `jpike`. O link foi feito para **uma pessoa numa empresa**. Cite: P2 parte HTML + esta decodificação.

## 3. Tabela de indicadores, já desarmada

Desarmar (*defang*) é trocar `.` por `[.]` e `http` por `hxxp`, para ninguém clicar sem querer e nenhum sistema transformar o texto em link. A função abaixo faz isso.

In [ ]:
def desarmar(valor):
    """Troca pontos por [.] e http por hxxp. Idempotente: não desarma duas vezes."""
    if "[.]" in valor:
        return valor
    return valor.replace("https://", "hxxps://").replace("http://", "hxxp://").replace(".", "[.]")

INDICADORES = [   # (indicador, tipo, onde, primeira observação UTC, notas)
    ("castellan-hardwood.example", "Domínio", "P2 From, Return-Path", "2026-03-10 14:02", "Imitação com hífen"),
    ("198.51.100.23", "IPv4", "P2 Received mais baixo", "2026-03-10 14:02", "Submissor autenticado"),
    ("203.0.113.47", "IPv4", "P2 Received do meio", "2026-03-10 14:02", "Servidor de e-mail da imitação"),
    ("dwhitlock.castellan@freemail.example", "E-mail", "P2 Reply-To; P3 M6", "2026-03-10 14:02", "Webmail gratuito"),
    ("dana.whitlock@castellan-hardwood.example", "E-mail", "P2 From", "2026-03-10 14:02", "Remetente falso"),
    ("ar@castellan-hardwood.example", "E-mail", "P2 Authenticated sender; P3 M3", "2026-03-10 14:02", "Conta de submissão; endereço de remessa"),
    ("555-0148", "Telefone", "P2 assinatura; P3 carta", "2026-03-10 14:02", "VoIP (P6 L9)"),
    ("20260310140231.2C4A1E03@mail.castellan-hardwood.example", "Message-ID", "P2", "2026-03-10 14:02", ""),
    ("20260303144702.GA11873@mail.castellanhardwood.example", "In-Reply-To", "P2", "2026-03-10 14:02", "Message-ID do M1 real"),
    ("https://cstl-docshare.example/view/r", "URL", "P2 href", "2026-03-10 14:02", "Página de coleta de credenciais"),
    ("cstl-docshare.example", "Domínio", "P2 href", "2026-03-10 14:02", ""),
    ("https://castellanhardwood.example/img/logo.png", "URL", "P2 HTML img", "2026-03-10 14:02", "Logo do site REAL (hotlink)"),
    ("Pellbrook Savings & Trust", "Banco", "P3 M3 carta", "2026-03-11 13:52", "Banco da conta falsa"),
    ("conta final 8830", "Conta", "P3 M3 carta; M7", "2026-03-11 13:52", "Conta recebedora"),
    ("5d0c7a1e9b43f26c8e7a0d51b9f4c3e2a6d8b17f04e9c5a2d3b6f8e1c7a9042b", "SHA-256", "P3 M3; P7", "2026-03-11 13:52", "Castellan_Bank_Letter_2026.pdf"),
    ("20260312163958.8E71B2C4@mail.castellan-hardwood.example", "Message-ID", "P7 (M6)", "2026-03-12 16:40", "Follow-up do atacante"),
]
SO_TEXTO = {"Banco", "Conta", "SHA-256", "Telefone"}        # não precisam de desarme
linhas = ["| # | Indicador | Tipo | Onde | Primeira obs. (UTC) | Notas |", "|---|---|---|---|---|---|"]
for i, (ind, tipo, onde, quando, notas) in enumerate(INDICADORES, 1):
    valor = ind if tipo in SO_TEXTO else desarmar(ind)
    linhas.append(f"| {i} | {valor} | {tipo} | {onde} | {quando} | {notas} |")
print("\n".join(linhas))
print(f"\n{len(INDICADORES)} indicadores (checkpoint: pelo menos 15)")

## 4. O site real × o clone (`difflib`, o `diff -u` do Python)

In [ ]:
import difflib

def bloco_citado(texto):
    """Pega só as linhas renderizadas (as que começam com '> ') e o head html."""
    return [l.lstrip("> ").rstrip() for l in texto.splitlines()
            if (l.startswith(">") or l.startswith("<")) and "SYNTHETIC" not in l]   # ignora o aviso

real = bloco_citado(PACOTE["04-vendor-site-archived.md"])
# No clone, pegamos só a parte B1 (antes de "## B2")
clone_txt = PACOTE["05-lookalike-site-capture.md"].split("## B2")[0]
clone = bloco_citado(clone_txt)

for linha in difflib.unified_diff(real, clone, "A-site-real", "B1-clone", lineterm="", n=0):
    print(linha)

Leia o `diff`: linhas com `-` só existem no site real; com `+` só no clone. Procure: a seção *Payments* sumiu, a *Banking update* apareceu, a CFO sumiu da equipe, telefone e e-mails mudaram, o `v=2024.1` do CSS, o © 2024. E a página **B2** (privacidade) tem uma frase que menciona **outra empresa**: abaixo.

In [ ]:
b2 = PACOTE["05-lookalike-site-capture.md"].split("## B2")[1].split("## B3")[0]
for frase in re.split(r"(?<=\.)\s+", " ".join(bloco_citado(b2))):
    if "Castellan" not in frase and frase.strip():
        print("Frase que não é da Castellan:", frase)

A frase *"Marrowline Freight Services respects your privacy..."* é **sobra de um clone anterior**. Ela vale um pivô inteiro: leva ao `marrowline-docs.example` do L1/L3/L6.

## 5. IPs: dono e tipo de rede, nunca o prefixo

O original avisa: pelo menos um IP está no mesmo /24 de infraestrutura sem relação. A célula agrupa os IPs do L8 por /24 e mostra que a faixa `198.51.100.0/24` tem donos completamente diferentes.

In [ ]:
import ipaddress
from collections import defaultdict

l8 = PACOTE["06-lookup-results.md"].split("## L8")[1].split("## L9")[0]
por_rede = defaultdict(list)
for linha in l8.splitlines():
    cols = [x.strip() for x in linha.strip("|").split("|")]
    if len(cols) >= 4 and re.fullmatch(r"\d+\.\d+\.\d+\.\d+", cols[0]):
        rede = ipaddress.ip_network(cols[0] + "/24", strict=False)   # o /24 que contém o IP
        por_rede[str(rede)].append((cols[0], cols[1], cols[2], cols[3]))

for rede, ips in por_rede.items():
    donos = {asn for _, asn, _, _ in ips}
    print(f"\n{rede}  ->  {len(ips)} IPs, {len(donos)} dono(s) diferente(s)")
    for ip, asn, dono, tipo in ips:
        print(f"   {ip:15} {asn:8} {dono:28} {tipo}")

`198.51.100.23` (VPS Pinecrate, atacante) está no mesmo /24 do celular do Jordan (`.77`, operadora móvel) e dos servidores de e-mail da Castellan (`.170`, `.180`). **Prefixo igual, nada em comum.**

## 6. 🎮 Quiz dos pivôs: achado, pista ou rejeitado?

Responda **A** (achado: dois tipos independentes ou vínculo direto confirmado), **P** (pista: um só tipo de pivô) ou **R** (rejeitado: a explicação inocente explica). Os 5 primeiros são do capstone; os 9 seguintes são os pivôs V1–V9 da Operação Isca.

In [ ]:
# Dados sintéticos da Operação Isca deste dia (plano B embutido).
ISCA_EMBUTIDO = {
"pivos-isca.csv": "# FICTÍCIO — Operação Isca (investigador-cyber-90-dias). Nenhuma pessoa, conta, número ou domínio real.\nid,entidade_a,entidade_b,valor_compartilhado,tipo_pivo,fonte,explicacao_inocente\nV1,serraazul-pix.example,serraazul-seguranca.example,mesmo certificado TLS (SAN com os dois nomes),certificado,CT 2026-10-04,nenhuma: um certificado so com os dois nomes foi pedido por quem controla os dois\nV2,serraazul-pix.example,serraazul-seguranca.example,A 203.0.113.77,IP,DNS passivo,203.0.113.77 e VPS de inquilino unico (3 nomes)\nV3,serraazul-pix.example,coopserra-azul.example,script /static/ik.js com o mesmo SHA-256,recurso da pagina,varredura de URL,o arquivo nao e biblioteca publica conhecida\nV4,serraazul-pix.example,coopvale-pix.example,script /static/ik.js com o mesmo SHA-256,recurso da pagina,varredura de URL publica (relato de outra cooperativa),o arquivo nao e biblioteca publica conhecida\nV5,serraazul-pix.example,coopvale-pix.example,comentario 'isca v2 - pescador' no codigo,texto no codigo,varredura de URL publica,comentario pode ser copiado por quem compra o kit\nV6,serraazul-pix.example,lojinha-doces.example,encurtador lnk.example,encurtador,SMS da ouvidoria,lnk.example e encurtador publico com milhoes de links\nV7,coopserra-azul.example,pousada-ipe.example,IP 198.51.100.200,IP,DNS passivo,198.51.100.200 e hospedagem compartilhada com 2.300 dominios\nV8,serraazul-pix.example,coopserra-azul.example,mesmo registrador barato e DNS gratuito,registro,WHOIS,registrador com milhoes de dominios e DNS gratuito com dezenas de milhares de zonas\nV9,serraazul-pix.example,serraazul.example,favicon com o mesmo hash,favicon,busca de favicon,o clone copiou o favicon da cooperativa real\n"
}

In [ ]:
import urllib.request

RAW_ISCA = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-84/"     # pasta caso-continuo/dados/dia-84 no GitHub

ISCA = {}
for nome, copia in ISCA_EMBUTIDO.items():
    try:                                                     # plano A: versão publicada no repo
        with urllib.request.urlopen(RAW_ISCA + nome, timeout=10) as r:
            ISCA[nome] = r.read().decode("utf-8")
        print(f"{nome}: baixado do GitHub")
    except Exception:                                        # plano B: cópia embutida
        ISCA[nome] = copia
        print(f"{nome}: cópia embutida")

In [ ]:
import csv, io

PIVOS = [
    ("🇨🇦", "cstl-docshare e portal.marrowline-docs carregam o mesmo vw-portal.min.js (mesmo SHA-256)", "P",
     "Forte, mas é um tipo só (recurso de página). Vira achado quando outro tipo converge (frase da B2, IP .23)."),
    ("🇨🇦", "cstl-docshare divide 203.0.113.88 com ferncastle-bakery e mais 1.411 domínios", "R",
     "Hospedagem compartilhada não liga nada (L5, L8)."),
    ("🇨🇦", "cstl-docshare, marrowline e Brightquay Legal têm o mesmo favicon mmh3 -1742093313", "R",
     "Brightquay usa o modelo público OpenDocViewer, outro JS, certificados desde 2021."),
    ("🇨🇦", "198.51.100.23 foi mail.marrowline-docs até 9/mar e submeteu o phishing em 10/mar", "A",
     "Somado ao JS idêntico e à frase da B2: tipos independentes convergem. Considere a reatribuição de IP como rival."),
    ("🇨🇦", "Todos os domínios do atacante usam Quillfeather Names + hollowdns", "P",
     "Fraco sozinho: registrador barato com 3 milhões de domínios e DNS gratuito com 90 mil zonas."),
]
# Os nove pivôs do Isca vêm do CSV; o gabarito fica aqui, separado do dado.
GABARITO_ISCA = {
    "V1": ("A", "Certificado com os dois nomes só existe se quem controla os dois pediu (vínculo direto)."),
    "V2": ("P", "VPS de inquilino único: forte, e casa com V1, mas sozinho é um tipo só."),
    "V3": ("P", "Script idêntico, não é biblioteca pública: forte, um tipo só."),
    "V4": ("P", "Mesmo script contra a Coopvale: fonte de terceiro, um tipo só."),
    "V5": ("P", "O comentário está DENTRO do mesmo script: não é tipo independente de V3/V4."),
    "V6": ("R", "Encurtador público com milhões de links."),
    "V7": ("R", "Hospedagem compartilhada com 2.300 domínios."),
    "V8": ("P", "Registrador barato e DNS gratuito: fraquíssimo sozinho."),
    "V9": ("R", "O clone copiou o favicon da vítima: não liga o real ao golpe."),
}
linhas_csv = [l for l in ISCA["pivos-isca.csv"].splitlines() if not l.startswith("#")]
for row in csv.DictReader(io.StringIO("\n".join(linhas_csv))):
    certo, porque = GABARITO_ISCA[row["id"]]
    PIVOS.append(("🇧🇷", f"{row['id']}: {row['entidade_a']} × {row['entidade_b']} — {row['valor_compartilhado']} "
                        f"(explicação inocente: {row['explicacao_inocente']})", certo, porque))

MODO_JOGO = True
acertos = 0
for i, (b, desc, certo, porque) in enumerate(PIVOS, 1):
    print(f"\n[{i:02d}/{len(PIVOS)}] {b} {desc}")
    if MODO_JOGO:
        r = input("   A, P ou R? ").strip().upper()[:1]
        acertos += (r == certo)
        print("   ✅" if r == certo else f"   ❌ era {certo}")
    print(f"   {certo} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(PIVOS)} (Checkpoint BR: pelo menos 10)")

## 7. 📝 Nota de pivô e "Não coletado"

Use o formulário para escrever a sua nota de pivô para a ponte Serra Azul–Coopvale e a nota de parada.

In [ ]:
#@title Notas do reconhecimento { display-mode: "form" }
valor_compartilhado = "Script /static/ik.js com o mesmo SHA-256"  #@param {type:"string"}
tipo_pivo = "Recurso de página (um tipo; o comentário 'pescador' está no mesmo arquivo)"  #@param {type:"string"}
explicacao_inocente = "Biblioteca pública? Não: o arquivo não aparece em sites sem relação"  #@param {type:"string"}
veredito = "Pista forte, um só tipo de pivô. Falta um segundo tipo (certificado, IP de inquilino único)"  #@param {type:"string"}
nao_coletado = "Estado ao vivo do IscaKit (proibido); dados de quem contratou o VPS (só por ordem judicial); titulares das chaves Pix (canal da cooperativa)"  #@param {type:"string"}

from pathlib import Path
Path("notas").mkdir(exist_ok=True)
nota = (f"# Recon — Dia 84\n\n## Nota de pivô: Serra Azul × Coopvale\n"
        f"- Valor compartilhado: {valor_compartilhado}\n- Tipo: {tipo_pivo}\n"
        f"- Explicação inocente checada: {explicacao_inocente}\n- Veredito: {veredito}\n\n"
        f"## Não coletado\n{nao_coletado}\n\n## Indicadores\n" + "\n".join(linhas) + "\n")
Path("notas/recon-dia84.md").write_text(nota, encoding="utf-8")
print(nota[:600], "...")
try:
    from google.colab import files
    files.download("notas/recon-dia84.md")
except ImportError:
    print("(Fora do Colab: salvo em notas/recon-dia84.md)")

## ✅ Antes de fechar o Dia 84

- [ ] Saltos `Received` lidos de baixo para cima; frase sobre o que o SPF prova e não prova
- [ ] Parâmetros `u` e `t` decodificados e registrados como achado de direcionamento
- [ ] Tabela com 15+ indicadores desarmados e citados
- [ ] Pelo menos seis diferenças entre o site real e o clone, incluindo a frase da B2
- [ ] Quiz dos pivôs com pelo menos 10/14

**Amanhã:** Dia 85: recebimento e planejamento. As perguntas do cliente viram perguntas que a evidência consegue responder, e as hipóteses entram numa matriz ACH.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>